In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import torch
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('../dataset/mushrooms.csv')

In [3]:
df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


## Data Quality

### Missing Values

In [4]:
missing_df = df.isnull().sum().reset_index()
missing_df.columns = ['Column', 'Missing Count']
missing_df['Missing Percentage'] = (
    missing_df['Missing Count'] / len(df) * 100
)

fig = px.line(
    missing_df,
    x='Column',
    y='Missing Count',
    hover_data='Missing Percentage'
)

fig.update_traces(
    hovertemplate = 
    "<b> %{x} </b> <br>" +
    "Missing Count = %{y} <br>" +
    "Missing Percentage = %{customdata[0]:.2f}%"
)
fig.show()

### Duplicate Rows

In [5]:
df.duplicated().sum()

np.int64(0)

## Unique Values

In [6]:
unique_counts = {}
for col in df.columns:
    unique_counts[col] = df[col].nunique()

unique_df = pd.DataFrame(
    unique_counts.items(),
    columns = ['Column', 'Unique Count']
)

fig = px.bar(
    unique_df,
    x = 'Column',
    y = 'Unique Count'
)
fig.show()

## Encoding

### Binary Encoding

In [7]:
categories_list = []
binary_columns = []
ohe_columns = []

for key, value in unique_counts.items():
    if value==2:
        categories_list.append(df[key].unique())
        binary_columns.append(key)
    else:
        ohe_columns.append(key)


In [8]:
binary_encoder = OrdinalEncoder(categories=categories_list)
df[binary_columns] = binary_encoder.fit_transform(df[binary_columns])

In [9]:
df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,0.0,x,s,n,0.0,p,0.0,0.0,0.0,k,...,s,w,w,p,w,o,p,k,s,u
1,1.0,x,s,y,0.0,a,0.0,0.0,1.0,k,...,s,w,w,p,w,o,p,n,n,g
2,1.0,b,s,w,0.0,l,0.0,0.0,1.0,n,...,s,w,w,p,w,o,p,n,n,m
3,0.0,x,y,w,0.0,p,0.0,0.0,0.0,n,...,s,w,w,p,w,o,p,k,s,u
4,1.0,x,s,g,1.0,n,0.0,1.0,1.0,k,...,s,w,w,p,w,o,e,n,a,g


### One-hot Encoding

In [10]:
df = pd.get_dummies(df, columns=ohe_columns, dtype=int)

In [11]:
df.head()

,class,bruises,gill-attachment,gill-spacing,gill-size,stalk-shape,cap-shape_b,cap-shape_c,cap-shape_f,cap-shape_k,...,population_s,population_v,population_y,habitat_d,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
1,1.0,0.0,0.0,0.0,1.0,0.0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,1.0,0.0,0.0,0.0,1.0,0.0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
4,1.0,1.0,0.0,1.0,1.0,1.0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


## Train Test Split

In [12]:
X = df.drop(columns=['class'])
y = df['class']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    stratify=y
)

## Visualizing Data Classes Using PCA

In [15]:
pca = PCA(n_components=2)
x_pca = pca.fit_transform(X)
pca_df = pd.DataFrame(
    x_pca,
    columns = ['C1', 'C2']
)

pca_fig = px.scatter(
    pca_df,
    x = 'C1',
    y = 'C2',
    color=y
)
pca_fig.show()
pca_fig.write_image("../visualizations/binary_class/class_distribution_pca.png")

## Implementation with PyTorch

In [16]:
class LogisticRegression:

    def __init__(self, lr=0.01, iterations=1000):
        self.lr = lr
        self.iterations = iterations
        self.loss_history = []


    def binary_cross_entropy(self, y_hat, y):
        eps = 1e-8
        return -(y * torch.log(y_hat + eps) + (1-y) * torch.log(1-y_hat + eps)).mean()

    def fit(self, X, y):
        m, n = X.shape

        X = torch.tensor(X.to_numpy(), dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        self.weights = torch.zeros(n, requires_grad=True)
        self.bias = torch.zeros(1, requires_grad=True)

        for i in range(self.iterations):
            z = X @ self.weights + self.bias
            y_hat = z.sigmoid()

            loss = self.binary_cross_entropy(y_hat, y)
            self.loss_history.append(loss.item())
            loss.backward()

            with torch.no_grad():
                self.weights -= self.lr * self.weights.grad
                self.bias -= self.lr * self.bias.grad
                
            self.weights.grad.zero_()
            self.bias.grad.zero_()

    def predict(self, X):
        X = torch.tensor(X.to_numpy(), dtype=torch.float32)

        with torch.no_grad():
            z = X @ self.weights + self.bias
            prob = z.sigmoid()

        return (prob>=0.5).numpy()

In [17]:
model = LogisticRegression()

In [18]:
model.fit(X_train, y_train)

In [19]:
preds = model.predict(X_test)

In [20]:
preds

array([False,  True, False, ...,  True,  True, False], shape=(2031,))

## Performance

In [21]:
cm = confusion_matrix(y_test, preds)

In [22]:
print(cm)

[[ 871  108]
 [  11 1041]]


In [23]:
fig = px.line(
    model.loss_history
)
fig.show()

## Decision Boundary in PCA Space

In [24]:
model.fit(pca_df, y)

In [25]:
# Get learned parameters
w = model.weights.detach().numpy()
b = model.bias.item()

# x values spanning the existing PCA plot
x_line = np.linspace(
    pca_df["C1"].min(),
    pca_df["C1"].max(),
    300
)

y_line = -(w[0] * x_line + b) / w[1]

mask = (
    (y_line >= pca_df["C2"].min()) &
    (y_line <= pca_df["C2"].max())
)

pca_fig.add_scatter(
    x=x_line[mask],
    y=y_line[mask],
    mode="lines",
    line=dict(color="black", width=3),
    name="Decision Boundary"
)
pca_fig.show()
pca_fig.write_image('../visualizations/binary_class/decision_boundary_pca.png')

## Using Sklearn 

In [26]:
sklearn_model = LogisticRegression()

In [27]:
sklearn_model.fit(X_train, y_train)

In [177]:
sklearn_preds = sklearn_model.predict(X_test)

In [179]:
cm = confusion_matrix(y_test, sklearn_preds)

In [184]:
print(cm)

[[ 979    0]
 [   0 1052]]
